#Statistics for Decision Making

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('property.csv')
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df.head()

,suburb,address,rooms,type,price,method,sellerg,date,distance,postcode,...,bathroom,car,landsize,buildingarea,yearbuilt,councilarea,lattitude,longtitude,regionname,propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


## 1. Altona Hypothesis Test

In [2]:
altona = df[df['suburb'] == 'Altona']['price']
t_stat, p_value = stats.ttest_1samp(altona, 800000)
altona.mean(), t_stat, p_value

(np.float64(834830.4054054054),
 np.float64(1.0277020770199676),
 np.float64(0.307483271305555))

## 2. Seasonal Price Difference

In [3]:
df_2016 = df[df['date'].dt.year == 2016]
winter = df_2016[df_2016['date'].dt.month.isin([10,11,12,1,2,3])]['price']
summer = df_2016[~df_2016['date'].dt.month.isin([10,11,12,1,2,3])]['price']
t_stat, p_value = stats.ttest_ind(winter, summer)
t_stat, p_value

(np.float64(4.043386317851058), np.float64(5.3309767667631686e-05))

## 3. Probability (No Car Space)

In [4]:
abbotsford = df[df['suburb'] == 'Abbotsford']
p_no_car = (abbotsford['car'] == 0).mean()
from scipy.stats import binom
prob = binom.pmf(3, 10, p_no_car)
round(prob, 3)

np.float64(0.26)

## 4. Probability of 3 Rooms

In [5]:
round((abbotsford['rooms'] == 3).mean(), 3)

np.float64(0.357)

## 5. Probability of 2 Bathrooms

In [6]:
round((abbotsford['bathroom'] == 2).mean(), 3)

np.float64(0.339)

## 6. Richmond Hypothesis Test

In [7]:
richmond = df[df['suburb'] == 'Richmond']['price']
t_stat, p_value = stats.ttest_1samp(richmond, 1000000)
richmond.mean(), t_stat, p_value

(np.float64(1083564.423076923),
 np.float64(2.579547704074923),
 np.float64(0.01044499066415202))

## 7. Car vs No Car Test

In [8]:
with_car = df[df['car'] > 0]['price']
no_car = df[df['car'] == 0]['price']
t_stat, p_value = stats.ttest_ind(with_car, no_car)
t_stat, p_value

(np.float64(-0.22347786369074255), np.float64(0.8231669871217725))

## 8. Two-Way ANOVA

In [9]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
model = ols('price ~ C(suburb) + C(type) + C(suburb):C(type)', data=df).fit()
sm.stats.anova_lm(model, typ=2)

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 313, but rank is 150
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 2, but rank is 0
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1923: RuntimeWarning: invalid value encountered in divide
  F /= J
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 626, but rank is 375
  warnings.warn('covariance of constraints does not have full '


,sum_sq,df,F,PR(>F)
C(suburb),3.559152e+15,313.0,70.854970,0.000000e+00
C(type),NaN,2.0,NaN,NaN
C(suburb):C(type),6.470034e+14,626.0,6.440216,2.505093e-263
Residual,2.068639e+15,12890.0,NaN,NaN


## 10. Bathrooms Policy Test

In [10]:
more_bath = df[df['bathroom'] > 2]['price']
less_bath = df[df['bathroom'] <= 2]['price']
t_stat, p_value = stats.ttest_ind(more_bath, less_bath)
t_stat, p_value

(np.float64(46.02604887995347), np.float64(0.0))